In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

C:\Users\49498\AppData\Roaming\Python\Python312\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [2]:
# Load dataset
url = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
df = pd.read_csv(url)
df['Country/Region'] = df['Country/Region'].str.strip().str.lower()
df = df.drop(columns=['Province/State', 'Lat', 'Long']).dropna()
df_us = df[df['Country/Region'] == 'us']

In [3]:
# Process the data to get daily cases and create columns
us = df_us.sum().reset_index().iloc[1:]
us.columns = ['Date', 'Cumulative_Cases']
us['Date'] = pd.to_datetime(us['Date'])
us['Daily_Cases'] = pd.to_numeric(us['Cumulative_Cases']).diff().fillna(0)

# Y(t) is actually t, Daily_Cases is the Y(t)
first_case_date = us[us['Daily_Cases'] > 0].index[0]
us['Y(t)'] = us.index - first_case_date

# Create lag features for Y(t) and Z(t)
for i in range(1, 6):
    us[f'Y(t-{i})'] = us['Daily_Cases'].shift(i)

us['Z(t)'] = (us['Daily_Cases'] > us['Daily_Cases'].shift(1)).astype(int)
for i in range(1, 6):
    us[f'Z(t-{i})'] = us['Z(t)'].shift(i)

us_lag = us.copy().dropna()

C:\Users\49498\AppData\Local\Temp\ipykernel_22680\3562381167.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  us['Date'] = pd.to_datetime(us['Date'])


In [4]:
# Data preparation for neural networks
x_linear = us_lag[['Y(t-1)', 'Y(t-2)', 'Y(t-3)', 'Y(t-4)', 'Y(t-5)']].values.astype(np.float32)
y_linear = us_lag['Daily_Cases'].values.astype(np.float32).reshape(-1, 1)

x_logit = us_lag[['Z(t-1)', 'Z(t-2)', 'Z(t-3)', 'Z(t-4)', 'Z(t-5)']].values.astype(np.float32)
y_logit = us_lag['Z(t)'].values.astype(np.float32).reshape(-1, 1)

# Standardize the x and y for better training
x_linear_mean, x_linear_std = x_linear.mean(axis=0), x_linear.std(axis=0)
y_linear_mean, y_linear_std = y_linear.mean(), y_linear.std()

x_logit_mean, x_logit_std = x_logit.mean(axis=0), x_logit.std(axis=0)

x_linear = (x_linear - x_linear_mean) / x_linear_std
y_linear = (y_linear - y_linear_mean) / y_linear_std
x_logit = (x_logit - x_logit_mean) / x_logit_std

# Change to tensors
x_linear_tensor = torch.tensor(x_linear)
y_linear_tensor = torch.tensor(y_linear)
x_logit_tensor = torch.tensor(x_logit)
y_logit_tensor = torch.tensor(y_logit)

In [5]:
# Define neural network model
def train_model(model, criterion, x_t, y_t, lr, epochs=1000):
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # dataset is not large, so use the whole dataset as a batch
    dataset = TensorDataset(x_t, y_t)
    dataloader = DataLoader(dataset, batch_size=len(x_t), shuffle=True)

    for epoch in range(epochs):
        model.train()
        for batch_x, batch_y in dataloader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
    model.eval()
    with torch.no_grad():
        final_loss = criterion(model(x_t), y_t).item()
    return final_loss

In [6]:
# Model training
model_linear = nn.Sequential(nn.Linear(5, 1))
loss_linear_scaled = train_model(model_linear, nn.MSELoss(), x_linear_tensor, y_linear_tensor, lr=0.01)
loss_linear = loss_linear_scaled * (y_linear_std ** 2)

model_logit = nn.Sequential(nn.Linear(5, 1))
loss_logit = train_model(model_logit, nn.BCEWithLogitsLoss(), x_logit_tensor, y_logit_tensor, lr=0.01)

In [7]:
# Report coefficients in original scale
w_linear_scaled = model_linear[0].weight.detach().numpy().flatten()
b_linear_scaled = model_linear[0].bias.detach().numpy()[0]

w_linear_unscaled = w_linear_scaled * (y_linear_std / x_linear_std)
b_linear_unscaled = y_linear_mean + b_linear_scaled * y_linear_std - np.sum(w_linear_unscaled * x_linear_mean)

equation_linear = f"Y(t) = {b_linear_unscaled:.4f}"
for i, w in enumerate(w_linear_unscaled, 1):
    equation_linear += f" + {w:.4f}*Y(t-{i})"
print(f"MSE: {loss_linear:.4f}")
print(f"Linear: {equation_linear}")

w_logit_scaled = model_logit[0].weight.detach().numpy().flatten()
b_logit_scaled = model_logit[0].bias.detach().numpy()[0]

w_logit_unscaled = w_logit_scaled / x_logit_std
b_logit_unscaled = b_logit_scaled - np.sum(w_logit_unscaled * x_logit_mean)

print('')
equation_logit = f"Z(t) = {b_logit_unscaled:.4f}"
for i, w in enumerate(w_logit_unscaled, 1):
    equation_logit += f" + {w:.4f}*Z(t-{i})"
print(f"BCEWithLogitsLoss: {loss_logit:.4f}")
print(f"Logistic: {equation_logit}")

MSE: 4116668160.0000
Linear: Y(t) = 6393.3516 + 0.5184*Y(t-1) + -0.0104*Y(t-2) + 0.2197*Y(t-3) + 0.0988*Y(t-4) + 0.1043*Y(t-5)

BCEWithLogitsLoss: 0.6464
Logistic: Z(t) = 1.3343 + -0.5068*Z(t-1) + -0.1688*Z(t-2) + -0.8333*Z(t-3) + -1.0309*Z(t-4) + -0.2180*Z(t-5)


In [8]:
# Model training with DNN
model_linear_dnn = nn.Sequential(
    nn.Linear(5, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1)
)
loss_linear_dnn_scaled = train_model(model_linear_dnn, nn.MSELoss(), x_linear_tensor, y_linear_tensor, lr=0.01)
loss_linear_dnn = loss_linear_dnn_scaled * (y_linear_std ** 2)

model_logit_dnn = nn.Sequential(
    nn.Linear(5, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1)
)
loss_logit_dnn = train_model(model_logit_dnn, nn.BCEWithLogitsLoss(), x_logit_tensor, y_logit_tensor, lr=0.01)

In [9]:
# Calculate error rates
with torch.no_grad():
    y_linear_true = y_linear_tensor * y_linear_std + y_linear_mean
    pred_linear = model_linear(x_linear_tensor) * y_linear_std + y_linear_mean
    # Weighted Mean Absolute Percentage Error
    err_linear = (torch.sum(torch.abs(pred_linear - y_linear_true)) / torch.sum(y_linear_true)) * 100
    
    pred_linear_dnn = model_linear_dnn(x_linear_tensor) * y_linear_std + y_linear_mean
    err_linear_dnn = (torch.sum(torch.abs(pred_linear_dnn - y_linear_true)) / torch.sum(y_linear_true)) * 100
    
    pred_logit_prob = torch.sigmoid(model_logit(x_logit_tensor))
    pred_logit_class = (pred_logit_prob >= 0.5).float()
    err_logit = torch.mean(torch.abs(pred_logit_class - y_logit_tensor)) * 100
    
    pred_logit_dnn_prob = torch.sigmoid(model_logit_dnn(x_logit_tensor))
    pred_logit_dnn_class = (pred_logit_dnn_prob >= 0.5).float()
    err_logit_dnn = torch.mean(torch.abs(pred_logit_dnn_class - y_logit_tensor)) * 100

In [10]:
final_table = pd.DataFrame({
    'Model Type': [
        'Linear (No Hidden)', 
        'Linear DNN (2 Hidden)', 
        'Logit (No Hidden)', 
        'Logit DNN (2 Hidden)',
    ],
    'Loss Function': [
        'MSE', 
        'MSE', 
        'BCE', 
        'BCE'
    ],
    'Error rates': [
        f"{err_linear.item():.2f}", 
        f"{err_linear_dnn.item():.2f}", 
        f"{err_logit.item():.2f}", 
        f"{err_logit_dnn.item():.2f}"
    ]
})
print(final_table.to_string(index=False))

           Model Type Loss Function Error rates
   Linear (No Hidden)           MSE       35.57
Linear DNN (2 Hidden)           MSE       14.14
    Logit (No Hidden)           BCE       36.12
 Logit DNN (2 Hidden)           BCE       32.34
